OSU AI 535, Winter 2026
Group: Jacob Beitler, Jonathan Hotchkiss, Seth Mackovjak

Overview:
1) Data preprocessing: Import, structure, split, and augment the data.
2) Model definition: ResNetRS50 binary categorization.
3) Training loop definition: Incorporate appropriate Weights and Bias.
3) Model saving.

In [ ]:
# Import ALL_IDB2 dataset and build a binary ResNet50 model
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights

from dataset_builder import idb2_dataset, category_balance_plot

# Image preprocessing (ResNet50 expects 224x224 RGB input normalized like ImageNet)
transform = T.Compose([
    T.ConvertImageDtype(torch.float32),
    T.Resize((224, 224)),
    T.Normalize(mean=ResNet50_Weights.IMAGENET1K_V2.transforms.mean,
                std=ResNet50_Weights.IMAGENET1K_V2.transforms.std),
])

# Create train/test Datasets with transforms applied
train_dataset = idb2_dataset(train=True, transform=transform)
test_dataset = idb2_dataset(train=False, transform=transform)

category_balance_plot(train_dataset=train_dataset,
                      test_dataset=test_dataset)

# Build a binary ResNet50 model (output logits for a single class)
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 1)

# Use BCEWithLogitsLoss for binary classification
criterion = nn.BCEWithLogitsLoss()

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model built:", model)
print("Device:", device)


In [ ]:
# Helper to build ResNet models with different depths for experimentation
from torchvision.models import (
    resnet18, resnet34, resnet50, resnet101, resnet152,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    ResNet101_Weights, ResNet152_Weights,
)

_resnet_builders = {
    "resnet18": (resnet18, ResNet18_Weights),
    "resnet34": (resnet34, ResNet34_Weights),
    "resnet50": (resnet50, ResNet50_Weights),
    "resnet101": (resnet101, ResNet101_Weights),
    "resnet152": (resnet152, ResNet152_Weights),
}


def build_resnet_binary(model_name: str = "resnet50", pretrained: bool = True, device=None) -> nn.Module:
    """Build a binary classification ResNet model.

    Args:
        model_name: One of resnet18/34/50/101/152.
        pretrained: If True, loads Imagenet weights.
        device: torch device (e.g. torch.device("cuda") or torch.device("cpu")).

    Returns:
        A torch.nn.Module configured for binary output (1 logit).
    """

    if model_name not in _resnet_builders:
        raise ValueError(f"Unsupported model: {model_name}. Choose from {list(_resnet_builders)}")

    builder, weights_cls = _resnet_builders[model_name]
    weights = weights_cls.IMAGENET1K_V2 if pretrained else None
    model = builder(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)

    if device is not None:
        model = model.to(device)

    return model


# Example: build multiple models for comparison
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models_to_try = ["resnet18", "resnet34", "resnet50", "resnet101", "resnet152"]
built_models = {name: build_resnet_binary(name, pretrained=True, device=device) for name in models_to_try}

print("Built models:", list(built_models.keys()))
print("Example model summary (resnet50):", built_models["resnet50"])
